# Exploración inicial de los datos

In [1]:
import pandas as pd

PATH = "paysim/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(
    PATH,
    usecols=["step", "nameOrig", "isFraud"]
)

lengths = df.groupby("nameOrig").size()
sender_labels = df.groupby("nameOrig")["isFraud"].max()

print("Transacciones:", len(df))
print("Remitentes únicos:", df["nameOrig"].nunique())
print("\nDistribución de transacciones por remitente:")
print(lengths.describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]))

print("\nRemitentes por longitud:")
for minimum in [1, 2, 3, 5, 10, 20]:
    print(f"{minimum:2d} o más: {(lengths >= minimum).sum():,}")

fraud_senders = sender_labels[sender_labels == 1].index
fraud_lengths = lengths.loc[fraud_senders]

print("\nRemitentes fraudulentos:", len(fraud_senders))
print("Longitud de remitentes fraudulentos:")
print(fraud_lengths.describe())

print("\nFraudulentos por longitud:")
for minimum in [1, 2, 3, 5, 10]:
    print(f"{minimum:2d} o más: {(fraud_lengths >= minimum).sum():,}")

Transacciones: 6362620
Remitentes únicos: 6353307

Distribución de transacciones por remitente:
count    6.353307e+06
mean     1.001466e+00
std      3.832002e-02
min      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
90%      1.000000e+00
95%      1.000000e+00
99%      1.000000e+00
max      3.000000e+00
dtype: float64

Remitentes por longitud:
 1 o más: 6,353,307
 2 o más: 9,298
 3 o más: 15
 5 o más: 0
10 o más: 0
20 o más: 0

Remitentes fraudulentos: 8213
Longitud de remitentes fraudulentos:
count    8213.000000
mean        1.003409
std         0.058293
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         2.000000
dtype: float64

Fraudulentos por longitud:
 1 o más: 8,213
 2 o más: 28
 3 o más: 0
 5 o más: 0
10 o más: 0


99.85 % de los remitentes tiene una sola transacción.
Solo 9,313 remitentes tienen más de una.
La longitud máxima es 3.
Solo 28 remitentes fraudulentos tienen dos transacciones.
Ningún remitente fraudulento alcanza tres transacciones.

Con estos datos, una LSTM no podría aprender frecuencia, cambios de destino ni evolución temporal. Hacer padding sobre secuencias de longitud uno produciría una arquitectura secuencial solo de nombre.

In [2]:
from pathlib import Path
import pandas as pd

BASE = Path("ibm-transactions")

files = [
    BASE / "HI-Small_Trans.csv",
    BASE / "LI-Small_Trans.csv",
]

for path in files:
    sample = pd.read_csv(path, nrows=10)

    print("=" * 70)
    print("Archivo:", path.name)
    print(f"Tamaño: {path.stat().st_size / 1024**2:,.2f} MB")
    print("Columnas:")
    print(sample.columns.tolist())
    print("\nTipos inferidos:")
    print(sample.dtypes)
    print("\nPrimeras filas:")
    print(sample.head(10).to_string(index=False))

Archivo: HI-Small_Trans.csv
Tamaño: 453.63 MB
Columnas:
['Timestamp', 'From Bank', 'Account', 'To Bank', 'Account.1', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

Tipos inferidos:
Timestamp                 str
From Bank               int64
Account                   str
To Bank                 int64
Account.1                 str
Amount Received       float64
Receiving Currency        str
Amount Paid           float64
Payment Currency          str
Payment Format            str
Is Laundering           int64
dtype: object

Primeras filas:
       Timestamp  From Bank   Account  To Bank Account.1  Amount Received Receiving Currency  Amount Paid Payment Currency Payment Format  Is Laundering
2022/09/01 00:20         10 8000EBD30       10 8000EBD30          3697.34          US Dollar      3697.34        US Dollar   Reinvestment              0
2022/09/01 00:20       3208 8000F4580        1 8000F5340             0.01          US 